# Check Image quality and shift for all camera

2026.01.05: Health Check before the PFS run

The engineering fibers were used to measure focal plane and check image quality. Because the fiber size is smaller, you cannot use EE5, instead, use EE3.

The following IIS images are used for each camera

* b: Kr 60s and Ne 1s
* r: Ne 1s
* n: Kr 60s
* m: Ne 1s

In [1]:
%load_ext autoreload
%autoreload 2

## Import, Setting

In [2]:
from pfs.lam.opdb import *
from pfs.lam.imqual2Csv import main as getImqual
from pfs.lam.bestFocusPlanefromCsv import main as getBestFocus

from lsst.daf.butler import Butler

In [3]:
from multiprocessing import Pool
import time
import os
import numpy as np
import glob

In [4]:
# To read data manually
from pfs.lam.linePeaksList import filterPeakList
import astropy.io.fits as fio
from pfs.lam.analysisPlot import plotRoiPeak, plotPeaksBrightness
from pfs.lam.detAnalysis import *

### DRP folder

In [5]:
# Hilo
drpPath, repo, rerun = '/work/datastore', 'repo', 'drpActor/reductions'
#drpPath, repo, rerun = '/work/datastore', 'repo', 'PFS/defaults'
datastore = drpPath
collection = [c for c in list(Butler(datastore).registry.queryCollections()) if c.startswith(rerun)]
drpVer='gen3'

### Data processing
Specify `arm`, `specId` and `visit_set_id` (or `experimentId`) for each dataset

#### Common settings to analysis

In [6]:
site = "Subaru"
fiberType = 'ENGINEERING'
outpath = "/work/moritani/spsAIT/202601/throughFocus/"

# Option to find peak
roi_size = 16 #24
seek_size = None
doBck = True

# measurement parameter
if fiberType == 'ENGINEERING':
    criteria = "EE3"
else:
    criteria = "EE5"
piston_index ="motor1"

# Option to define outputs, plots
roiPlot = True
plotPeaksFlux = True
doFit = True    # activate 2D-gaussian fit to measure spot size
doLSF = False
doPrint=False

In [7]:
# specfify the peaklist data
d_kr = {'b1': '20240419', 'b2': '20240419', 'b3': '20240419', 'b4': '20240419',
        'n1': '20240822', 'n2': '20250307', 'n3': '20240710', 'n4': '20240724'} 

d_ne = {'b1': '20240415', 'b2': '20240415', 'b3': '20240415', 'b4': '20240415',
        'r1': '20240415', 'r2': '20240415', 'r3': '20240415', 'r4': '20240415',
        'm1': '20251109', 'm2': '20251109', 'm3': '20251109', 'm4': '20251109'}

#### Check if processed data processed by drpActor (Hilo) exists

2026.01 exists

If the data exists, run the cells "By using bulter" If not run the cells "By readind processed image directly"

In [8]:
datastore = '/work/datastore'
collections = ['drpActor/reductions', 'PFS/defaults']

butler = Butler(datastore, collections=collections, instrument='PFS')
dataId=dict(visit=136265, spectrograph=1, arm='b')
butler.get("postISRCCD", dataId)

By reading processed image directly

(2026.01.07) Because process by drpActor in Hilo is slow, use image at the summit.

In [203]:
# NIR, Kr
#experimentId = 57135  # visit_sequence_id
#visitId = 136433
#experimentId = 57140  # visit_sequence_id
#visitId = 136488
#experimentId = 57142  # visit_sequence_id
#visitId = 136490
# New position
#experimentId = 57161  # visit_sequence_id
#visitId = 136503
# update pfs_instdata
#experimentId = 57165  # visit_sequence_id
#visitId = 136507
# monitor after the second adjustment
#experimentId = 57218  # visit_sequence_id
#visitId = 136629
#experimentId = 57548  # visit_sequence_id
#visitId = 136779
experimentId = 57703  # visit_sequence_id
visitId = 136986
arm = "n"
lamps = ['Kr']
line = 'Kr'
exptime = 60
colldir = '20251119T205640Z'

# Red, Ne
#experimentId = 57136  # visit_sequence_id
#visitId = 136434
#experimentId = 57141  # visit_sequence_id
#visitId = 136489
#experimentId = 57166  # visit_sequence_id
#visitId = 136508
# monitor after the second adjustment
#experimentId = 57219  # visit_sequence_id
#visitId = 136630
#experimentId = 57549  # visit_sequence_id
#visitId = 136781
experimentId = 57704  # visit_sequence_id
visitId = 136987
arm = "r"
lamps = ['Ne']
line = 'Ne'
exptime = 1
colldir = '20251119T205640Z'

# Blue, Kr
#experimentId = 57135  # visit_sequence_id
#visitId = 136433
#experimentId = 57140  # visit_sequence_id
#visitId = 136488
#experimentId = 57165  # visit_sequence_id
#visitId = 136507
# monitor after the second adjustment
#experimentId = 57218  # visit_sequence_id
#visitId = 136629
#experimentId = 57548  # visit_sequence_id
#visitId = 136779
experimentId = 57703  # visit_sequence_id
visitId = 136986
arm = "b"
lamps = ['Kr']
line = 'Kr'
exptime = 60
colldir = '20251119T205640Z'

# Blue, Ne
#experimentId = 57136  # visit_sequence_id
#visitId = 136434
#experimentId = 57141  # visit_sequence_id
#visitId = 136489
# monitor after the second adjustment
#experimentId = 57219  # visit_sequence_id
#visitId = 136630
#experimentId = 57549  # visit_sequence_id
#visitId = 136781
experimentId = 57704  # visit_sequence_id
visitId = 136987
arm = "b"
lamps = ['Ne']
line = 'Ne'
exptime = 1
colldir = '20251119T205640Z'


#visitStart, visitEnd = getVisitRange_fromWeb(experimentId, url="http://133.40.164.16/sps-logs/index.html")
#print(f"{experimentId}: {visitStart} -- {visitEnd}")
#visitId = visitStart


#for specId in range(1,5):
for specId in [1]:

    cam = f"{arm}{specId}"
    imname =f'/data/drp/datastore/drpActor/reductions/{colldir}/postISRCCD/*/{visitId}/postISRCCD_PFS_{visitId}_{cam}_drpActor_reductions_{colldir}.fits'
    fname = glob.glob(imname)[0]

    csvPath = os.path.join(outpath,f"sm{specId}",f"Exp{experimentId}",rerun,f"roi{roi_size}",f"doBck{doBck}")

    if not os.path.exists(csvPath):
        os.makedirs(csvPath)


    print(f'Processing {cam}')
    if line == 'Kr':
        peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_kr_{cam}_{d_kr[cam]}_iis.csv"
    elif line == 'Ne':
        peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_ne_{cam}_{d_ne[cam]}_iis.csv"

    peaks = filterPeakList(peaklist, arm, lamps) if peaklist is not None else None

    fits = fio.open(fname)

    dataId=dict(arm=arm, spectrograph=specId, visit=visitId)
    imageInfo = dict(dataId)
    imageInfo.update(filename=fname)
    imageInfo.update(experimentId=experimentId)    

    # Image HDU1
    image = fits[1].data

    if roiPlot and (peaklist is not None):
        RoiPlotTitle = f"Peaklist roiPlot {cam.upper()} Exp{experimentId} - visit{visitId} - roi_size={roi_size}\n"
        plotRoiPeak(image, peaks, roi_size=roi_size, savePlotFile=os.path.join(csvPath,f"{cam}_{visitId}_rawPeak"),raw=True,doSave=True, title=RoiPlotTitle )

    
    data = getFullImageQuality(image, peaklist, imageInfo=imageInfo,
                               roi_size=roi_size, EE=[3,5], seek_size=seek_size,
                               com=True, doBck=doBck, doFit=doFit, doLSF=doLSF, doSep=True,fullSep=False,
                               doPlot=roiPlot, doPrint=doPrint,
                               mask_size=20, threshold=50, subpix=5 , maxPeakDist=80,
                               maxPeakFlux=40000, minPeakFlux=2000, calexpMask=None)

    now = datetime.now() # current date and time\n",
    date_time = now.strftime("%Y%m%dT%Hh%M")
    lwh=None

    csvName = f"Imquality_{cam}_Exp{experimentId}_{visitId}_{date_time}.csv"
    if not os.path.exists(csvPath):
        os.makedirs(csvPath,exist_ok =True)
    data.to_csv(os.path.join(csvPath, csvName))
    if roiPlot:
        RoiPlotName = f"roiPlot_{cam}_Exp{experimentId}_{visitId}_{date_time}"
        RoiPlotTitle = f"roiPlot {cam.upper()} Exp{experimentId} - visitId {visitId} - roi_size={roi_size}\n{date_time}"
        plotRoiPeak(image, data, roi_size, savePlotFile=os.path.join(csvPath, RoiPlotName),raw=False,doSave=True, title=RoiPlotTitle)
    if plotPeaksFlux:
        plotPeaksBrightness(data, doSave=True, savePlotFile=os.path.join(csvPath,f"{cam}_{visitId}_fluxes_{'_'.join(lamps)}{exptime:.0f}s_lwh{lwh}"),
                            plot_title=f"{cam}_{visitId} - {'_'.join(lamps)} exptime {exptime}s lwh {lwh}")

Processing b1


By using butler

In [173]:
# Blue
# Kr, 60s
#experimentId = 56834  # visit_sequence_id
#visitId = 136268
# SM1 slit start and keep on
#experimentId = 56841  # visit_sequence_id
#visitId = 136278
# SM1 slit with new slit position
#experimentId = 56845  # visit_sequence_id
#visitId = 136332
# SM1 slit with new slit position, updated pfs_instdata
#experimentId = 56847  # visit_sequence_id
#visitId = 136336
experimentId = 57135  # visit_sequence_id
visitId = 136433
arms = ["b"]
line = 'Kr'

# Ne, 1s
#experimentId = 56836  # visit_sequence_id
#visitId = 136272
# SM1 slit start and keep on
#experimentId = 56842  # visit_sequence_id
#visitId = 136280
# SM1 slit with new slit position
#experimentId = 56846  # visit_sequence_id
#visitId = 136334
# SM1 slit with new slit position, updated pfs_instdata
#experimentId = 56848  # visit_sequence_id
#visitId = 136338
#experimentId = 57135  # visit_sequence_id
#visitId = 136433
#arms = ["b"]
#line = 'Ne'

# Red
# Ne, 1s
#experimentId = 56836
#visitId = 136272
# SM1 slit start and keep on
#experimentId = 56842  # visit_sequence_id
#visitId = 136280
# SM1 slit with new slit position
#experimentId = 56846  # visit_sequence_id
#visitId = 136334
# SM1 slit with new slit position, updated pfs_instdata
#experimentId = 56848  # visit_sequence_id
#visitId = 136338
#experimentId = 56851  # visit_sequence_id
#visitId = 136344
#arms = ["r"]
#line = 'Ne'

# Nir
# Kr, 60s
#experimentId = 56834  # visit_sequence_id
#visitId = 136268
# SM1 slit start and keep on
#experimentId = 56841  # visit_sequence_id
#visitId = 136278
# SM1 slit with new slit position
#experimentId = 56845  # visit_sequence_id
#visitId = 136332
# SM1 slit with new slit position, updated pfs_instdata
#experimentId = 56847  # visit_sequence_id
#visitId = 136336
#arms = ["n"]
#line = 'Kr'

# Red -MR
# Ne, 1s
#experimentId = 56838  # visit_sequence_id
#visitId = 136274
#arms = ["m"]
#line = 'Ne'


#visitStart, visitEnd = getVisitRange_fromWeb(experimentId, url="http://133.40.164.16/sps-logs/index.html")
#print(f"{experimentId}: {visitStart} -- {visitEnd}")
#visitId = visitStart

#for specId in range(1,5):
for specId in [1]:
    for arm in arms:

        cam = f"{arm}{specId}"
        print(f'Processing {cam}')
        if line == 'Kr':
            peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_kr_{cam}_{d_kr[cam]}_iis.csv"
        elif line == 'Ne':
            peaklist = f"/work/moritani/spsAIT/202311/peaklist/SM{specId}_peakList_ne_{cam}_{d_ne[cam]}_iis.csv"
        else:
            print(f"I don't analyze {line} now")
            continue
        getImqual(visitId, peaklist, cam, rerun, experimentId, outpath, drpPath, repo,
                  roi_size, seek_size, doBck, roiPlot, plotPeaksFlux, doFit, doLSF, doPrint, fiberType=fiberType,
                  drpVer=drpVer)

Processing b1


DatasetNotFoundError: Dataset postISRCCD with data ID {instrument: 'PFS', arm: 'b', spectrograph: 1, visit: 136433} could not be found in collections ('drpActor/reductions/20241113T171734Z', 'drpActor/reductions/20241113T173152Z', 'drpActor/reductions/20241204T184750Z', 'drpActor/reductions/20241204T185356Z', 'drpActor/reductions/20241204T185633Z', 'drpActor/reductions/20241204T185815Z', 'drpActor/reductions/20241205T204827Z', 'drpActor/reductions/20241205T204923Z', 'drpActor/reductions/20241205T204930Z', 'drpActor/reductions/20241205T205023Z', 'drpActor/reductions/20241205T205042Z', 'drpActor/reductions/20241205T205236Z', 'drpActor/reductions/20241205T205620Z', 'drpActor/reductions/20241205T210608Z', 'drpActor/reductions/20241205T211032Z', 'drpActor/reductions/20241205T211128Z', 'drpActor/reductions/20241205T211210Z', 'drpActor/reductions/20241205T213159Z', 'drpActor/reductions/20250109T214417Z', 'drpActor/reductions/20250122T093808Z', 'drpActor/reductions/20250122T100146Z', 'drpActor

Outputs (example of m arm):

Directory: `outpath`/sm1/Exp56838/drpActor/reductions/roi16/doBckTrue/
    
    ([outpath]/sm[specId]/Exp[experimentId]/drpActor/reductions/roi[roi_size]/doBck[doBck]/)

* m1_136274_rawPeak_roi_all.png : spots centering to position in the peaklist
* m1_136274_fluxes_Ne1s_lwhNone.png  : flux of measured lines
* roiPlot_m1_Exp56838_136274_20260105T13h02_roi_all.png : spots centering to the measured peak position
* Imquality_m1_Exp56838_136274_20260105T13h02.csv measured data

Note that measured spots include false detection (bad pixel etc) too.

Record the file path of csv file to `/work/moritani/spsAIT/camsrecord.yaml`

## plot result

In [8]:
import pandas as pd
from matplotlib import pyplot as plt
import yaml

In [9]:
def plot_spotsize_twopanels(xs, ys, dx, vmin1, vmax1, dy, vmin2, vmax2,
                            xmin=-250, xmax=250, ymin=-250, ymax=250,
                            title='', titlesuf=['x', 'y'], fname='plot', cmap='viridis'):

    # set the font sizes for labels
    plt.rc('xtick', labelsize=10)
    plt.rc('ytick', labelsize=10)

    # scatter plot, with or without ragne limit
    fig = plt.figure(figsize=(8, 3), dpi=90, facecolor='w', edgecolor='k')
    ax1 = fig.add_axes((0.1, 0.15, 0.35, 0.75), aspect='equal')
    ax2 = fig.add_axes((0.6, 0.15, 0.35, 0.75), aspect='equal')

    # x and y axis is the same between the two panels
    ax1.set_xlim(xmin=xmin, xmax=xmax)
    ax1.set_ylim(ymin=ymin, ymax=ymax)
    ax1.set_title(titlesuf[0], fontsize=10)
    ax2.set_xlim(xmin=xmin, xmax=xmax)
    ax2.set_ylim(ymin=ymin, ymax=ymax)
    ax2.set_title(titlesuf[1], fontsize=10)

    sc1 = ax1.scatter(xs, ys, c=dx, vmin=vmin1, vmax=vmax1, marker="o",
                      cmap=cmap, lw=0)
    sc2 = ax2.scatter(xs, ys, c=dy, vmin=vmin2, vmax=vmax2, marker="o",
                      cmap=cmap, lw=0)

    xlname = "X"
    ylname = "Y"

    plt.colorbar(sc1, ax=ax1, label='[pix]')
    plt.colorbar(sc2, ax=ax2, label='[pix]')
    ax1.set_xlabel(xlname, fontsize=10)
    ax1.set_ylabel(ylname, fontsize=10)
    ax2.set_xlabel(xlname, fontsize=10)
    ax2.set_ylabel(ylname, fontsize=10)
    #plt.savefig(fname+".pdf")
    #plt.show()
    fig.suptitle(title, fontsize=10)

    return fig

In [10]:
with open('/work/moritani/spsAIT/camsrecord.yaml', 'rb') as f:
    yml = yaml.safe_load(f)

In [11]:
# dataset  to analyze
date = 202601

In [12]:
yml[date]['b1']['Kr'].split('_')

['/work/moritani/spsAIT/202601/throughFocus/sm1/Exp56834/drpActor/reductions/roi16/doBckTrue/Imquality',
 'b1',
 'Exp56834',
 '136268',
 '20260105T13h58.csv']

### b arm

####  b1

In [18]:
cam = 'b1'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile2.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
df1 = df1[(df1.brightness<=50000) & (df1.wavelength!=640.40177)] #& (df1.wavelength!=446.49427) & (df1.wavelength!=557.18362) & (df1.wavelength!=587.25432)& (df1.wavelength!=609.78506)   & (df1.wavelength!=638.4756)]  #  

In [19]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley],cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')

####  b2

In [20]:
cam = 'b2'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = pd.read_csv(iqfile2)
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=640.40177) &(df1.fwhm<10) ]  
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=585.41101) & (df1.wavelength!=587.25432) & (df1.wavelength!=594.6481) & (df1.wavelength!=640.40177) & (df1.wavelength!=638.4756)& (df1.wavelength!=609.78506)& (df1.wavelength!=607.60193)]  #  

In [21]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')

####  b3

In [22]:
cam = 'b3'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = pd.read_csv(iqfile2)
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=585.41101) & (df1.wavelength!=587.25432) & (df1.wavelength!=594.6481) & (df1.wavelength!=640.40177) & (df1.wavelength!=638.4756)& (df1.wavelength!=609.78506)& (df1.wavelength!=607.60193)]  #  
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=640.40177) &(df1.fwhm<10) ]  

In [24]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 =plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')

####  b4

In [25]:
cam = 'b4'
# Kr
iqfile1 = yml[date][cam]['Kr']
visit1 = iqfile1.split('_')[-2]
# Ne
iqfile2 = yml[date][cam]['Ne']
visit2 = iqfile1.split('_')[-2]

df1 = pd.concat([pd.read_csv(iqfile1), pd.read_csv(iqfile2)], ignore_index=True)
#df1 = pd.read_csv(iqfile2)
#df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=585.41101) & (df1.wavelength!=587.25432) & (df1.wavelength!=594.6481) & (df1.wavelength!=640.40177) & (df1.wavelength!=638.4756)& (df1.wavelength!=609.78506)& (df1.wavelength!=607.60193)]  #  
df1 = df1[(df1.brightness<=50000) &(df1.wavelength!=640.40177) &(df1.fwhm<10) ]  

In [26]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit2}),Kr({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit2})_Kr({visit1}).png')

### r arm

####  r1

In [27]:
cam = 'r1'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000)]

In [28]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

####  r2

In [29]:
cam = 'r2'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[df1.wavelength!=724.71631] #df1.brightness<=50000]

In [30]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

####  r3

In [31]:
cam = 'r3'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[df1.wavelength!=724.71631] #df1.brightness<=50000]

In [32]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

####  r4

In [33]:
cam = 'r4'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[df1.wavelength!=724.71631] #df1.brightness<=50000]

In [34]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

### n arm

####  n1

In [38]:
cam = 'n1'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=55000) & (df1.fwhm<=10) & (df1.wavelength!=1182.26136) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [37]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')

####  n2

In [39]:
cam = 'n2'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=60000) & (df1.fwhm<=10) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [40]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
#print(f'x:{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f}, y:{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f}')

fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')

####  n3

In [42]:
cam = 'n3'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=55000) & (df1.fwhm<=10) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [43]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')

####  n4

In [45]:
cam = 'n4'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.brightness<=55000) & (df1.fwhm<=10) & ((df1.sep_flag==0)|(df1.sep_flag==2))]

In [46]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Kr({visit1}).png')

### m arm

####  m1

In [47]:
cam = 'm1'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000)]

In [48]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

####  m2

In [50]:
cam = 'm2'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000)]

In [51]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

####  m3

In [52]:
cam = 'm3'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000)]

In [53]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

####  m4

In [54]:
cam = 'm4'
iqfile1 = yml[date][cam]
visit1 = iqfile1.split('_')[-2]

# You need to exclude satulated spots or too faint spots by specifying brightnss, wavelength, etc.
df1 = pd.read_csv(iqfile1)
df1 = df1[(df1.wavelength!=724.71631) & (df1.brightness<=50000)]

In [55]:
vmin = 1.2 ; vmax = 2.5
title = f'FWHM: Ne({visit1}) {cam}'
titlex = f'{np.nanmean(df1.fwhm_x):.1f}+/-{np.nanstd(df1.fwhm_x):.1f} pix'
titley = f'{np.nanmean(df1.fwhm_y):.1f}+/-{np.nanstd(df1.fwhm_y):.1f} pix'
fig1 = plot_spotsize_twopanels(df1['oid_x'], df1['oid_y'], df1['fwhm_x'], vmin, vmax, df1['fwhm_y'], vmin, vmax,
                               xmin=0, xmax=4100, ymin=-0, ymax=4100, title=title, titlesuf=[titlex,titley], cmap='viridis')
fig1.savefig(f'{cam}_fwhm_Ne({visit1}).png')

## Spot shift

In [13]:
def plot_spotshift_twopanels(df, xrange=2., yrange=2., detx=4176, dety=4176, title=''):


    fig = "comp_pos"; plt.close(fig);
    fig, axs = plt.subplots(num=fig, nrows=1, ncols=2, sharex=False, sharey=False,
                            figsize=(8,3))

    # scatter  
    axs[0].scatter(df.oid_x_x-df.oid_x_y, df.oid_y_x-df.oid_y_y)
    dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
    dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
    axs[0].scatter(dx_mean, dy_mean)

    axs[0].set_xlim(xmin=-1*xrange, xmax=xrange)
    axs[0].set_ylim(ymin=-1*yrange, ymax=yrange)
    axs[0].set_aspect('equal')

    # quiver
    axs[1].quiver(df.oid_x_y, df.oid_y_y, df.oid_x_x-df.oid_x_y, df.oid_y_x-df.oid_y_y, color='dimgrey', scale=1e+2,label='before')

    axs[1].set_xlim(xmin=0., xmax=detx)
    axs[1].set_ylim(ymin=0., ymax=dety)
    axs[1].set_aspect('equal')

        
    fig.suptitle(title, fontsize=9)

    #plt.show()

    return fig

In [14]:
# Previous measurement
date_old='202511e'

### b arm

#### b1

In [17]:
cam = 'b1'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [18]:
# plot
xrange=5.
yrange=5.
detsize=4176
dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)
fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### b2

In [19]:
cam = 'b2'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [20]:
# plot
xrange=5.
yrange=5.
detsize=4176
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### b3

In [21]:
cam = 'b3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [22]:
# plot
xrange=5.
yrange=5.
detsize=4176
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### b4

In [25]:
cam = 'b4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]['Kr']
visit_b1 = iqfile_b1.split('_')[-2]
# Ne
iqfile_b2 = yml[date_old][cam]['Ne']
visit_b2 = iqfile_b2.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]['Kr']
visit_a1 = iqfile_a1.split('_')[-2]
# Ne
iqfile_a2 = yml[date][cam]['Ne']
visit_a2 = iqfile_a2.split('_')[-2]

df_b = pd.concat([pd.read_csv(iqfile_b1), pd.read_csv(iqfile_b2)])
df_b = df_b[(df_b.fwhm_y<10)]
df_b = df_b.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df_a = pd.concat([pd.read_csv(iqfile_a1), pd.read_csv(iqfile_a2)])
df_a = df_a[(df_a.fwhm_y<10)]
df_a = df_a.sort_values(by=['fiber', 'wavelength', 'oid_y'])

df = df_a.merge(df_b, on = ['fiber', 'wavelength'])

In [26]:
# plot
xrange=5.
yrange=5.
detsize=4176
title = f"Old: Ne({visit_b2}),Kr({visit_b1}), New: Ne({visit_a2}),Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

### r arm

#### r1

In [23]:
cam = 'r1'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [24]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### r2

In [27]:
cam = 'r2'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [28]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### r3

In [29]:
cam = 'r3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [30]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### r4

In [31]:
cam = 'r4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [32]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

### n arm

#### n1

In [33]:
cam = 'n1'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [34]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### n2

In [35]:
cam = 'n2'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [36]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### n3

In [37]:
cam = 'n3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [38]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### n4

In [39]:
cam = 'n4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [40]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Kr({visit_b1}), New: Kr({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

### m arm

#### m1

In [44]:
cam = 'm1'
# Old
# Ne
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Ne
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [42]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### m2

In [45]:
cam = 'm2'
# Old
# Ne
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Ne
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [46]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### m3

In [47]:
cam = 'm3'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [48]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')

#### m4

In [49]:
cam = 'm4'
# Old
# Kr
iqfile_b1 = yml[date_old][cam]
visit_b1 = iqfile_b1.split('_')[-2]

# New
# Kr
iqfile_a1 = yml[date][cam]
visit_a1 = iqfile_a1.split('_')[-2]

df_b1 = pd.read_csv(iqfile_b1)
df_a1 = pd.read_csv(iqfile_a1)

df = df_a1.merge(df_b1, on = ['fiber', 'wavelength'])

In [50]:
# plot
xrange=5.
yrange=5.
detsize=4176

dx_mean=np.nanmedian(df.oid_x_x-df.oid_x_y)
dy_mean=np.nanmedian(df.oid_y_x-df.oid_y_y)
title = f"Old: Ne({visit_b1}), New: Ne({visit_a1}) {cam}\nmedian:(dx,dy)=({dx_mean:.1f}, {dy_mean:.1f}) pix"
fig=plot_spotshift_twopanels(df, xrange=xrange, yrange=yrange, detx=detsize, dety=detsize, title=title)

fig.savefig(f'spotshift_{cam}_delta_{date}_{date_old}.png')